# Neuron Group PCA — 2D/3D Scatter Plots

The within-group variance analysis showed PC1+PC2+PC3 captures ~80–90% of group
variance while PC1 alone only reaches ~40%. This notebook examines the **shape** of
that 3D subspace:

- Are neurons in a group arranged on a **ring** in PC1–PC2 space (analogous to the
  torus geometry seen in centroids and parameter trajectories)?
- Do neurons **converge onto** the ring during second descent, or were they always arranged that way?
- Does the shape differ between models with clear graduation structure (p109) vs diffuse ones (p113, p101)?

All projections use a **reference basis fixed at the final epoch** so trajectories
across epochs are comparable. Sign flips are corrected by aligning each epoch's
eigenvectors to the reference.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader

In [2]:
FAMILY_NAME = "modulo_addition_1layer"
EXPORT_DIR = Path("exports/neuron_group_pca_scatter")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# (prime, model_seed, data_seed, label)
MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (59,  485, 598, "p59 incomplete freq set"),
    (101, 999, 598, "p101 high-acc no structure"),
]

family = load_family(FAMILY_NAME)

## PCA helpers

In [3]:
N_COMPONENTS = 3


def compute_basis(group_W):
    """SVD of centered group weight matrix.

    Args:
        group_W: (d_model, n_group) — W_in columns for the group

    Returns:
        basis: (N_COMPONENTS, d_model) — top-3 left singular vectors (row = PC)
        s:     (N_COMPONENTS,)         — singular values
        coords: (n_group, N_COMPONENTS) — projections of each neuron
    """
    centroid = group_W.mean(axis=1, keepdims=True)
    centered = group_W - centroid
    U, s, _ = np.linalg.svd(centered, full_matrices=False)
    basis = U[:, :N_COMPONENTS].T  # (N_COMPONENTS, d_model)
    coords = (basis @ centered).T  # (n_group, N_COMPONENTS)
    return basis, s[:N_COMPONENTS], coords


def align_basis(basis, reference):
    """Flip PC signs so each vector maximizes dot product with the reference."""
    aligned = basis.copy()
    for k in range(N_COMPONENTS):
        if np.dot(basis[k], reference[k]) < 0:
            aligned[k] *= -1
    return aligned


def project_onto(group_W, basis):
    """Project group W_in onto a fixed basis (from another epoch)."""
    centroid = group_W.mean(axis=1, keepdims=True)
    centered = group_W - centroid
    return (basis @ centered).T  # (n_group, N_COMPONENTS)


def load_group_data(variant, prime):
    """Load group assignments, reference basis, and per-epoch projections.

    Returns dict with:
        groups: list of dicts, one per frequency group:
            freq, members, ref_basis, ref_s, ref_coords
        epochs: sorted list of all available epochs
        loader: ArtifactLoader (for per-epoch loading)
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = sorted(loader.get_epochs("parameter_snapshot"))

    # Group assignments from final epoch neuron_freq_norm
    norm_data = loader.load_epoch("neuron_freq_norm", epochs[-1])
    norm_matrix = norm_data["norm_matrix"]  # (n_freq, d_mlp)
    dominant_freq = np.argmax(norm_matrix, axis=0)  # (d_mlp,)

    # Reference basis from final epoch W_in
    final_snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in_final = final_snap["W_in"]  # (d_model, d_mlp)

    groups = []
    for f in range(norm_matrix.shape[0]):
        members = np.where(dominant_freq == f)[0]
        if len(members) < 2:
            continue
        basis, s, coords = compute_basis(W_in_final[:, members])
        groups.append({
            "freq": f,
            "members": members,
            "ref_basis": basis,
            "ref_s": s,
            "ref_coords": coords,
            "norm_frac": norm_matrix[:, members],  # (n_freq, n_group) — for coloring
        })

    return {"groups": groups, "epochs": epochs, "loader": loader}


def get_representative_epochs(epochs, n=5):
    """Pick n approximately evenly-spaced epochs across training."""
    indices = np.round(np.linspace(0, len(epochs) - 1, n)).astype(int)
    return [epochs[i] for i in indices]

## PC1 vs PC2 scatter — one group per model, multiple epochs

Each panel shows one frequency group (the largest group for each model) projected
onto PC1–PC2 of the **final-epoch basis**. Five representative epochs are overlaid;
color encodes epoch (early = light, late = dark). A ring arrangement would indicate
neurons are sampling different phases of a shared periodic direction.

In [4]:
def render_pc12_scatter_epochs(model_data, group, n_epochs=5, title=""):
    """Scatter of PC1 vs PC2 at n_epochs evenly-spaced checkpoints.

    Each epoch is one trace, colored along a viridis gradient.
    Final epoch is highlighted with larger markers.
    """
    loader = model_data["loader"]
    all_epochs = model_data["epochs"]
    snap_epochs = get_representative_epochs(all_epochs, n_epochs)

    members = group["members"]
    ref_basis = group["ref_basis"]

    fig = go.Figure()

    colors = [f"rgba({int(30 + 220*(i/(n_epochs-1)))}, "
              f"{int(50 + 120*(1 - i/(n_epochs-1)))}, "
              f"{int(180 - 140*(i/(n_epochs-1)))}, 0.7)"
              for i in range(n_epochs)]

    for i, epoch in enumerate(snap_epochs):
        snap = loader.load_epoch("parameter_snapshot", epoch)
        W_in = snap["W_in"]
        coords = project_onto(W_in[:, members], ref_basis)  # (n_group, 3)

        is_final = (epoch == all_epochs[-1])
        fig.add_trace(go.Scatter(
            x=coords[:, 0].tolist(),
            y=coords[:, 1].tolist(),
            mode="markers",
            name=f"epoch {epoch}",
            marker=dict(
                color=colors[i],
                size=8 if is_final else 5,
                line=dict(width=0.5, color="white") if is_final else dict(width=0),
            ),
            hovertemplate=f"epoch={epoch}<br>PC1=%{{x:.3f}}<br>PC2=%{{y:.3f}}<extra></extra>",
        ))

    # Add final epoch ring guide
    snap = loader.load_epoch("parameter_snapshot", all_epochs[-1])
    coords_final = project_onto(snap["W_in"][:, members], ref_basis)
    r = np.sqrt(coords_final[:, 0]**2 + coords_final[:, 1]**2).mean()
    theta = np.linspace(0, 2 * np.pi, 100)
    fig.add_trace(go.Scatter(
        x=(r * np.cos(theta)).tolist(),
        y=(r * np.sin(theta)).tolist(),
        mode="lines",
        line=dict(color="rgba(0,0,0,0.15)", width=1, dash="dot"),
        showlegend=False,
        hoverinfo="skip",
    ))

    s = group["ref_s"]
    total_var = (s**2).sum()
    pc1_frac = s[0]**2 / total_var
    pc2_frac = s[1]**2 / total_var

    fig.update_layout(
        title=f"{title} — freq {group['freq']} (n={len(members)})<br>"
              f"<sup>PC1 {pc1_frac:.1%} | PC2 {pc2_frac:.1%} | basis fixed at final epoch</sup>",
        xaxis_title=f"PC1 ({pc1_frac:.1%})",
        yaxis_title=f"PC2 ({pc2_frac:.1%})",
        yaxis=dict(scaleanchor="x"),
        template="plotly_white",
        height=500,
        margin=dict(l=60, r=20, t=80, b=60),
        legend=dict(orientation="h", y=-0.15),
    )
    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    model_data = load_group_data(v, prime)

    # Pick the largest group
    largest = max(model_data["groups"], key=lambda g: len(g["members"]))

    fig = render_pc12_scatter_epochs(model_data, largest, n_epochs=5, title=label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_pc12_scatter.png"))
    print(f"  {label}: {len(model_data['groups'])} groups")

  p109 reference healthy: 3 groups


  p113 canon (diffuse): 4 groups


  p59 incomplete freq set: 2 groups


  p101 high-acc no structure: 4 groups


## All groups for each model — PC1 vs PC2 at final epoch

One panel per frequency group, final epoch only. All groups in one figure per model.
Color encodes the neuron's dominant-frequency purity (max frac_explained) — brighter
markers are more specialized neurons.

In [5]:
def render_all_groups_final(model_data, label):
    """One subplot per frequency group, PC1 vs PC2 at final epoch."""
    groups = model_data["groups"]
    loader = model_data["loader"]
    epochs = model_data["epochs"]

    final_snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in = final_snap["W_in"]

    n = len(groups)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    subplot_titles = [f"freq {g['freq']} (n={len(g['members'])})" for g in groups]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles,
                        horizontal_spacing=0.08, vertical_spacing=0.12)

    for idx, group in enumerate(groups):
        row, col = divmod(idx, cols)
        members = group["members"]
        coords = project_onto(W_in[:, members], group["ref_basis"])  # (n_group, 3)

        # Color by dominant-frequency purity
        purity = group["norm_frac"].max(axis=0)  # (n_group,)

        fig.add_trace(
            go.Scatter(
                x=coords[:, 0].tolist(),
                y=coords[:, 1].tolist(),
                mode="markers",
                marker=dict(
                    color=purity.tolist(),
                    colorscale="Viridis",
                    size=5,
                    showscale=(idx == 0),
                    colorbar=dict(title="purity", thickness=12) if idx == 0 else None,
                    cmin=0, cmax=1,
                ),
                showlegend=False,
                hovertemplate="PC1=%{x:.3f}<br>PC2=%{y:.3f}<extra></extra>",
            ),
            row=row + 1, col=col + 1,
        )

    fig.update_layout(
        title=f"{label} — all groups, PC1 vs PC2, final epoch<br>"
              "<sup>Color = dominant-frequency purity (frac_explained)</sup>",
        template="plotly_white",
        height=350 * rows,
        margin=dict(l=60, r=60, t=80, b=60),
    )
    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    model_data = load_group_data(v, prime)
    fig = render_all_groups_final(model_data, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_all_groups_final.png"))

## Neuron trajectory through PC1–PC2 space

For the largest group in p109 (the clearest model): each neuron's path through
PC1–PC2 space across all checkpoints. Subsampled to 40 neurons for readability.
Color encodes epoch progression (early = light blue, late = dark red).

If neurons converge onto a ring, we expect trajectories to spiral outward from
the center onto the ring boundary.

In [6]:
def render_neuron_trajectory(model_data, group, n_subsample=40, stride=1, title=""):
    """Trajectory of subsampled neurons through PC1–PC2 space over all epochs."""
    loader = model_data["loader"]
    all_epochs = model_data["epochs"]
    ref_basis = group["ref_basis"]
    members = group["members"]

    rng = np.random.default_rng(42)
    if len(members) > n_subsample:
        selected = rng.choice(len(members), n_subsample, replace=False)
    else:
        selected = np.arange(len(members))

    epochs_to_use = all_epochs[::stride]
    n_ep = len(epochs_to_use)

    # Build (n_ep, n_selected, 2) trajectory array
    trajectories = np.empty((n_ep, len(selected), 2))
    for i, epoch in enumerate(epochs_to_use):
        snap = loader.load_epoch("parameter_snapshot", epoch)
        coords = project_onto(snap["W_in"][:, members[selected]], ref_basis)
        trajectories[i] = coords[:, :2]

    fig = go.Figure()

    for n_idx in range(len(selected)):
        x = trajectories[:, n_idx, 0].tolist()
        y = trajectories[:, n_idx, 1].tolist()
        # Color gradient encoded as a single colored path
        fig.add_trace(go.Scatter(
            x=x, y=y,
            mode="lines",
            line=dict(width=0.8, color="rgba(100,120,200,0.25)"),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Mark start (circle) and end (diamond)
        fig.add_trace(go.Scatter(
            x=[x[0], x[-1]], y=[y[0], y[-1]],
            mode="markers",
            marker=dict(
                symbol=["circle", "diamond"],
                size=[5, 7],
                color=["rgba(120,180,255,0.6)", "rgba(220,60,60,0.8)"],
            ),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Reference ring
    final_coords = trajectories[-1]  # (n_selected, 2)
    r = np.sqrt(final_coords[:, 0]**2 + final_coords[:, 1]**2).mean()
    theta = np.linspace(0, 2 * np.pi, 100)
    fig.add_trace(go.Scatter(
        x=(r * np.cos(theta)).tolist(),
        y=(r * np.sin(theta)).tolist(),
        mode="lines",
        line=dict(color="rgba(0,0,0,0.1)", width=1, dash="dot"),
        showlegend=False,
        hoverinfo="skip",
    ))

    s = group["ref_s"]
    total_var = (s**2).sum()
    fig.update_layout(
        title=f"{title} — freq {group['freq']} (n={len(members)}, showing {len(selected)})<br>"
              f"<sup>Blue circle = epoch 0 | Red diamond = final | "
              f"PC1 {s[0]**2/total_var:.1%} | PC2 {s[1]**2/total_var:.1%}</sup>",
        xaxis_title="PC1",
        yaxis_title="PC2",
        yaxis=dict(scaleanchor="x"),
        template="plotly_white",
        height=550,
        margin=dict(l=60, r=20, t=80, b=60),
    )
    return fig


# Show trajectory for all four models, largest group each
for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    model_data = load_group_data(v, prime)
    largest = max(model_data["groups"], key=lambda g: len(g["members"]))
    fig = render_neuron_trajectory(model_data, largest, n_subsample=40, title=label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_trajectory.png"))

## 3D scatter — PC1 × PC2 × PC3 at final epoch

All groups overlaid in 3D, each group colored distinctly. Final epoch only.
Use the interactive Plotly 3D view to rotate and inspect the geometry.

In [7]:
import colorsys


def _group_color(g_idx, n_groups, alpha=0.6):
    hue = g_idx / max(n_groups, 1)
    r, g, b = colorsys.hls_to_rgb(hue, 0.55, 0.65)
    return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"


def render_3d_scatter(model_data, label):
    """3D scatter PC1×PC2×PC3 at final epoch, all groups colored distinctly."""
    loader = model_data["loader"]
    epochs = model_data["epochs"]
    groups = model_data["groups"]

    final_snap = loader.load_epoch("parameter_snapshot", epochs[-1])
    W_in = final_snap["W_in"]

    fig = go.Figure()

    for g_idx, group in enumerate(groups):
        members = group["members"]
        coords = project_onto(W_in[:, members], group["ref_basis"])  # (n_group, 3)
        color = _group_color(g_idx, len(groups))

        s = group["ref_s"]
        total_var = (s**2).sum()
        fracs = s[:3]**2 / total_var

        fig.add_trace(go.Scatter3d(
            x=coords[:, 0].tolist(),
            y=coords[:, 1].tolist(),
            z=coords[:, 2].tolist(),
            mode="markers",
            name=f"freq {group['freq']} (n={len(members)})",
            marker=dict(color=color, size=3),
            hovertemplate=(
                f"freq={group['freq']}<br>"
                "PC1=%{x:.3f}<br>PC2=%{y:.3f}<br>PC3=%{z:.3f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        title=f"{label} — PC1×PC2×PC3, final epoch, all groups",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
            aspectmode="cube",
        ),
        template="plotly_white",
        height=600,
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(font=dict(size=10)),
    )
    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    model_data = load_group_data(v, prime)
    fig = render_3d_scatter(model_data, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_3d_scatter.png"))